# 필라멘트 게이트 + 베드 마스킹 v2 확인 (전처리만)

모델 없이 게이트와 마스킹만 돌려 봅니다.
- 게이트: 출력물 면적 비율 0.1 미만이면 판정 보류
- 마스킹: 출력물 바깥(베드)을 회색으로 칠함. 출력물 윤곽 안쪽(언더필 틈 포함)은 원본 유지
- 계산 방법은 2번 셀 코드(`filament_gate.py`) 맨 위 설명 참고

**Drive `산공종설/`에 필요한 파일:** `underfill_originals_4000.zip` (선택: 본인 `share_pipeline_3models_20261002.zip`, 7번에서 사용)

GPU는 필요 없습니다. 1번부터 차례로 실행하면 2~3분이면 끝나고, 결과는 `산공종설/results/gate_check_v2/`에 저장됩니다. 게이트 코드는 2번 셀이 직접 만들어서 Drive에 다른 버전이 있어도 상관없습니다.

## 1. Drive 연결 + 파일 확인

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os, glob
DRIVE_DIR = "/content/drive/MyDrive/산공종설"   # 폴더 이름 다르면 여기만 바꾸기
OUT = f"{DRIVE_DIR}/results/gate_check_v2"
os.makedirs(OUT, exist_ok=True)
print("OK  " if os.path.exists(f"{DRIVE_DIR}/underfill_originals_4000.zip") else "없음", "underfill_originals_4000.zip")
DH_ZIP = next((p for p in [f"{DRIVE_DIR}/share_pipeline_3models_20261002.zip",
                           f"{DRIVE_DIR}/TalkFile_share_pipeline_3models_20261002.zip"] if os.path.exists(p)), None)
print("본인 zip:", DH_ZIP or "없음(선택)")

## 2. 게이트 코드 (filament_gate.py v2)
이 셀을 실행하면 `/content/filament_gate.py`가 만들어집니다. 팀원 코드에 넣을 때도 이 파일을 그대로 쓰면 됩니다.

In [ ]:
%%writefile /content/filament_gate.py
"""질감 기준 필라멘트 게이트 + 베드 마스킹 v2 (2026-10-02, 팀원)

하는 일 (고정 크롭 1000x1000 하나에 대해)
  1. 출력물 영역 찾기 -> 면적 비율(ratio)
  2. 게이트: ratio < 0.1 이면 판정 보류 (베드만 찍힌 사진)
  3. 마스킹: 출력물 바깥(베드)을 회색으로 칠한 크롭을 돌려줌 -> 모델 입력용
     (모델이 '반짝이 = 언더필'로 판단하는 문제를 막기 위함)

출력물 영역 찾는 순서 (250x250으로 줄여서 계산)
  a. 픽셀마다 주변 9x9 밝기 표준편차(질감)와 평균 밝기 계산
  b. 표준편차 < 10 (매끈) & 밝기 > 110 -> 출력물 후보
     (v1의 채도 조건은 삭제: 360장 시험에서 빼도 결과 동일, 자동으로 정하면 흰 출력물끼리 갈라버림)
  c. 7x7 열기·닫기, 화면 2% 미만 덩어리 버림
  d. 흔들린 베드 얼룩 제거: 덩어리의 (표준편차 중앙값 >= 5) & (밝기 중앙값 < 155) & (볼록도 < 0.8)
     -> 셋 다 해당하면 버림. 번진 반짝이는 덜 매끈하고 어둡고 모양이 들쭉날쭉함
  e. 남은 덩어리마다 바깥 윤곽 안쪽을 전부 채움 -> 언더필의 틈(결함 증거)이 회색으로 지워지지 않게
  f. 마스킹 때는 윤곽을 2픽셀(원본 기준 약 8픽셀) 넓혀서 테두리가 잘리지 않게 함

사용법
  from filament_gate import crop_roi, filament_ratio, gate, mask_bed
  crop = crop_roi(img)                     # 원본 1600x1200 -> 고정 크롭 1000x1000
  r = filament_ratio(crop)                 # RGB 기본 (본인 pipeline.py와 같은 형태)
  ok, r = gate(crop, bgr=True)             # cv2로 읽은 BGR이면 bgr=True. ok=False면 판정 보류
  masked, r = mask_bed(crop, bgr=True)     # 베드를 회색 칠한 크롭 (입력과 같은 크기·색 순서)

필요 패키지: numpy, opencv-python (Colab 기본 설치됨)
"""
import cv2
import numpy as np

VERSION = "v2-mask"
ROI = (0, 250, 1000, 1250)       # ymin, xmin, ymax, xmax
THRESHOLD = 0.1                  # 출력물 면적 이 비율 미만이면 판정 보류
SIZE, WIN = 250, 9               # 계산용 축소 크기, 질감 보는 창
SD_T, MEAN_T = 10, 110           # 출력물 후보: 매끈 & 밝음
MIN_BLOB = 0.02                  # 화면 2% 미만 덩어리 버림
BLUR_SD, BLUR_MEAN, BLUR_SOL = 5, 155, 0.8   # 흔들린 베드 얼룩 판정 (셋 다 해당하면 버림)
MARGIN = 2                       # 마스킹 여유 (축소 기준 픽셀)
GRAY_RGB = (124, 116, 104)       # ImageNet 평균색 -> 정규화하면 거의 0
_K7 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
_KM = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * MARGIN + 1, 2 * MARGIN + 1))


def crop_roi(img):
    """원본 프레임 -> 고정 크롭. 색 순서 상관없음."""
    y0, x0, y1, x1 = ROI
    assert img.shape[0] >= y1 and img.shape[1] >= x1, f"원본 크기가 크롭보다 작음: {img.shape}"
    return img[y0:y1, x0:x1]


def object_mask(crop, bgr=False):
    """출력물 영역 마스크 (SIZE x SIZE, 0/1, 윤곽 안 채움)."""
    crop = np.ascontiguousarray(crop, dtype=np.uint8)
    small = cv2.resize(crop, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    g = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY if bgr else cv2.COLOR_RGB2GRAY).astype(np.float32)
    m = cv2.blur(g, (WIN, WIN))
    sd = np.sqrt(np.maximum(cv2.blur(g * g, (WIN, WIN)) - m * m, 0))
    raw = ((sd < SD_T) & (m > MEAN_T)).astype(np.uint8)
    raw = cv2.morphologyEx(cv2.morphologyEx(raw, cv2.MORPH_OPEN, _K7), cv2.MORPH_CLOSE, _K7)
    n, lab, st, _ = cv2.connectedComponentsWithStats(raw)
    out = np.zeros_like(raw)
    for j in range(1, n):
        if st[j, cv2.CC_STAT_AREA] < MIN_BLOB * SIZE * SIZE:
            continue
        comp = lab == j
        cs, _ = cv2.findContours(comp.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        c = max(cs, key=cv2.contourArea)
        sol = cv2.contourArea(c) / max(cv2.contourArea(cv2.convexHull(c)), 1.0)
        if np.median(sd[comp]) >= BLUR_SD and np.median(m[comp]) < BLUR_MEAN and sol < BLUR_SOL:
            continue                                    # 흔들린 베드 얼룩
        cv2.drawContours(out, cs, -1, 1, thickness=-1)  # 바깥 윤곽 안쪽 전부 채움 (틈 유지)
    return out


def filament_ratio(crop, bgr=False):
    """출력물 면적 비율 (0~1). 본인 filament_ratio(rgb_crop)와 같은 형태."""
    return float(object_mask(crop, bgr).mean())


def gate(crop, threshold=THRESHOLD, bgr=False):
    """(통과 여부, 비율). 통과 못 하면 판정 보류."""
    r = filament_ratio(crop, bgr)
    return r >= threshold, r


def mask_bed(crop, bgr=False):
    """베드(출력물 바깥)를 회색으로 칠한 크롭과 비율. 크기·색 순서는 입력과 같음."""
    crop = np.ascontiguousarray(crop, dtype=np.uint8)
    om = object_mask(crop, bgr)
    keep = cv2.dilate(om, _KM)
    keep = cv2.resize(keep, (crop.shape[1], crop.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool)
    out = crop.copy()
    out[~keep] = GRAY_RGB[::-1] if bgr else GRAY_RGB
    return out, float(om.mean())


def overlay(crop, bgr=False, color=(255, 40, 40), alpha=0.55):
    """확인용: 출력물로 판단한 영역을 빨간색으로 칠한 RGB 이미지 (SIZE x SIZE)."""
    om = object_mask(crop, bgr)
    small = cv2.resize(np.ascontiguousarray(crop, dtype=np.uint8), (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    rgb = small[:, :, ::-1].copy() if bgr else small.copy()
    k = om == 1
    rgb[k] = ((1 - alpha) * rgb[k] + alpha * np.array(color)).astype(np.uint8)
    return rgb


def otsu_ratio(crop, bgr=False):
    """비교용: 기존 게이트 값 (팀원 A의 check_filament_presence와 같은 Otsu 밝은 픽셀 비율)."""
    g = cv2.cvtColor(np.ascontiguousarray(crop, dtype=np.uint8), cv2.COLOR_BGR2GRAY if bgr else cv2.COLOR_RGB2GRAY)
    _, b = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return float((b > 0).mean())


def _self_check():
    flat = np.full((1000, 1000, 3), 200, np.uint8)
    rng = np.random.default_rng(0)
    glitter = (rng.random((1000, 1000, 3)) > 0.7).astype(np.uint8) * 230
    assert filament_ratio(flat) > 0.95, filament_ratio(flat)
    assert filament_ratio(glitter) < 0.05, filament_ratio(glitter)
    half = glitter.copy(); half[500:] = 200
    mk, r = mask_bed(half)
    assert 0.4 < r < 0.6 and (mk[:400] == GRAY_RGB).all() and (mk[600:] == 200).all()


if __name__ == "__main__":
    # python filament_gate.py 사진.jpg ... -> 사진별 비율, 통과/보류, 기존 Otsu 값
    import sys
    _self_check()
    for p in sys.argv[1:]:
        img = cv2.imread(p)
        ok, r = gate(crop_roi(img), bgr=True)
        print(f"{p}\tratio={r:.3f}\t{'통과' if ok else '보류'}\t(otsu={otsu_ratio(crop_roi(img), bgr=True):.3f})")
    if len(sys.argv) == 1:
        print("self-check ok", VERSION)

## 3. 원본 풀기 + 게이트 불러오기
마지막 줄에 `4000장 | self-check ok | v2-mask`가 나와야 합니다.

In [ ]:
import sys, importlib
if len(glob.glob("/content/orig/**/*.jpg", recursive=True)) < 4000:
    !cp "{DRIVE_DIR}/underfill_originals_4000.zip" /content/
    !unzip -q -o /content/underfill_originals_4000.zip -d /content/orig
sys.path.insert(0, "/content")
import filament_gate as fg
importlib.reload(fg); fg._self_check()
print(len(glob.glob("/content/orig/**/*.jpg", recursive=True)), "장 | self-check ok |", fg.VERSION)

## 4. 4,000장 게이트 값 계산 (1~2분)
사진마다 새 비율(`new`)과 기존 Otsu 비율(`otsu`)을 `gate_values_v2.csv`로 저장합니다. 이미 있으면 불러오기만 합니다.

In [ ]:
import cv2, time, numpy as np, pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
paths = sorted(glob.glob("/content/orig/normal/*.jpg")) + sorted(glob.glob("/content/orig/underfill_50/*.jpg"))
gv_file = f"{OUT}/gate_values_v2.csv"
if os.path.exists(gv_file):
    gv = pd.read_csv(gv_file, encoding="utf-8-sig"); print("저장된 값 불러옴")
else:
    t0 = time.time()
    def one(p):
        crop = fg.crop_roi(cv2.imread(p))   # cv2라 BGR
        return fg.filament_ratio(crop, bgr=True), fg.otsu_ratio(crop, bgr=True)
    with ThreadPoolExecutor(4) as ex:
        vals = list(ex.map(one, paths))
    gv = pd.DataFrame({"source": [f"{Path(p).parent.name}/{Path(p).name}" for p in paths],
                       "label": [Path(p).parent.name for p in paths],
                       "frame": [int(Path(p).name[:8]) for p in paths],
                       "new": [round(v[0], 4) for v in vals], "otsu": [round(v[1], 4) for v in vals]})
    gv.to_csv(gv_file, index=False, encoding="utf-8-sig")
    print(f"완료: {time.time()-t0:.0f}초")
print(len(gv), "장")

## 5. 기존 vs 새 게이트: 보류 장수와 겹침
참고(v1 게이트, 4,000장): 529장 보류 (정상 257 / 언더필 272)

In [ ]:
import matplotlib.pyplot as plt
OLD_T, NEW_T = 0.25, fg.THRESHOLD
gv["old_held"] = gv.otsu < OLD_T
gv["new_held"] = gv.new < NEW_T
t = pd.DataFrame({"기존(Otsu<0.25)": gv.groupby("label").old_held.sum(), f"새 v2(<{NEW_T})": gv.groupby("label").new_held.sum()})
t.loc["합계"] = t.sum(); t.loc["전체 대비"] = (t.loc["합계"] / len(gv) * 100).round(1).astype(str) + "%"
print(t, "\n")
print(pd.Series({"둘 다 보류": (gv.old_held & gv.new_held).sum(), "기존만 보류": (gv.old_held & ~gv.new_held).sum(),
                 "새 게이트만 보류": (~gv.old_held & gv.new_held).sum(), "둘 다 통과": (~gv.old_held & ~gv.new_held).sum()}).to_string())
print("\n출력물 비율 중앙값:", gv.groupby("label").new.median().round(3).to_dict())
fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
for lab, d in gv.groupby("label"):
    ax[0].hist(d.otsu, bins=60, alpha=.5, label=lab); ax[1].hist(d.new, bins=60, alpha=.5, label=lab)
ax[0].axvline(OLD_T, c="k", ls="--"); ax[1].axvline(NEW_T, c="r", ls="--")
ax[0].set_title("old gate: Otsu bright ratio"); ax[1].set_title("new gate v2: object area ratio")
for a in ax: a.legend()
plt.tight_layout(); plt.savefig(f"{OUT}/gate_hist.png", dpi=100); plt.show()

## 6. 보류된 사진 (10장마다 1장)
클래스별 프레임 순서로 10장마다 1장. **빨간색 = 출력물로 판단한 영역**입니다. 정말 판정할 정보가 없는 사진인지 확인합니다. 그림은 `held_every10_p01.png` 등으로, 보류 목록은 `held_new_gate.csv`로 저장됩니다.

In [ ]:
STEP = 10
def show_grid(rows, title, fname, draw, ncol=6):
    for pg in range(0, len(rows), ncol * 4):
        chunk = rows.iloc[pg:pg + ncol * 4]; nrow = (len(chunk) + ncol - 1) // ncol
        fig, axs = plt.subplots(nrow, ncol, figsize=(2.6 * ncol, 2.9 * nrow), squeeze=False)
        for a in axs.flat: a.axis("off")
        for a, (_, r) in zip(axs.flat, chunk.iterrows()):
            a.imshow(draw(fg.crop_roi(cv2.imread(f"/content/orig/{r.source}"))))
            a.set_title(f"{r.label[:9]} #{r.frame}\nnew={r.new:.2f} otsu={r.otsu:.2f}", fontsize=8)
        fig.suptitle(f"{title} ({pg+1}-{pg+len(chunk)} / {len(rows)})")
        plt.tight_layout(); plt.savefig(f"{OUT}/{fname}_p{pg // (ncol*4) + 1:02d}.png", dpi=80); plt.show(); plt.close(fig)

held = gv[gv.new_held].sort_values(["label", "frame"])
held.to_csv(f"{OUT}/held_new_gate.csv", index=False, encoding="utf-8-sig")
sample = pd.concat([d.iloc[::STEP] for _, d in held.groupby("label")])
print(f"보류 {len(held)}장 중 {len(sample)}장 표시")
show_grid(sample, f"held, every {STEP}th", "held_every10", lambda c: fg.overlay(c, bgr=True))

## 7. 마스킹 결과 확인 (게이트 통과 사진)
모델에 들어가는 모습입니다. 게이트를 통과한 사진을 클래스별로 프레임 전체에서 고르게 12장씩 골라 보여 줍니다. 확인할 것:
- 베드(반짝이)가 회색으로 가려졌는지
- **언더필 출력물의 틈(줄무늬)이 지워지지 않고 남아 있는지**
- 출력물 일부가 회색으로 잘려 나가지 않았는지

그림은 `masked_check_p01.png` 등으로 저장됩니다.

In [ ]:
passed = gv[~gv.new_held].sort_values(["label", "frame"])
pick = pd.concat([d.iloc[np.linspace(0, len(d) - 1, 12).astype(int)] for _, d in passed.groupby("label")])
show_grid(pick, "masked model input (passed gate)", "masked_check", lambda c: fg.mask_bed(c, bgr=True)[0][:, :, ::-1])

## 8. (선택) 본인 "9번 모두 틀린 151장" 검증
본인 zip의 `gate_check_per_frame.csv`로, 게이트가 매번 틀린 사진을 얼마나 거르는지 본인 보고서 4장 표와 같은 형식으로 계산합니다.

읽는 법: "항상 맞힌 사진 중 보류"에는 **베드만 찍힌 언더필 사진**이 많이 들어갑니다. 이 사진들은 모델이 반짝이를 보고 우연히 맞힌 것이라(360장 시험에서 36장 전부 이 경우), 보류해도 손해로 보기 어렵습니다. 그래서 클래스별로 나눠서 보여 줍니다.

In [ ]:
import zipfile
if DH_ZIP is None:
    print("본인 zip이 없어서 건너뜀")
else:
    with zipfile.ZipFile(DH_ZIP) as z:
        name = next(n for n in z.namelist() if n.endswith("results/gate_check_per_frame.csv"))
        dh = pd.read_csv(z.extract(name, "/content/dh"), encoding="utf-8-sig")
    dh["source"] = dh.source.str.replace("\\", "/", regex=False)
    d = dh.merge(gv[["source", "new", "otsu"]].rename(columns={"otsu": "otsu_mine"}), on="source", how="left")
    print("매칭 안 된 사진:", d.new.isna().sum(), "| Otsu 값 차이 최대:", round((d.otsu_ratio - d.otsu_mine).abs().max(), 4))
    bad, good = d.wrong_of_9 == 9, d.wrong_of_9 == 0
    rows = []
    for name, col, th in [("기존 Otsu<0.25 (현재)", "otsu_mine", 0.25), ("기존 Otsu<0.45", "otsu_mine", 0.45),
                          ("기존 Otsu<0.50", "otsu_mine", 0.50), (f"새 v2<{NEW_T}", "new", NEW_T), ("새 v2<0.2", "new", 0.2)]:
        h = d[col] < th; h_all = gv["otsu" if col == "otsu_mine" else "new"] < th
        rows.append({"게이트": name,
                     f"매번 틀린 {bad.sum()}장 중 보류": f"{(h&bad).sum()}장 ({(h&bad).sum()/bad.sum():.0%})",
                     "항상 맞힘 중 보류: 정상": int((h & good & (d.label == "normal")).sum()),
                     "항상 맞힘 중 보류: 언더필": int((h & good & (d.label != "normal")).sum()),
                     "4,000장 중 보류": f"{h_all.sum()}장 ({h_all.mean():.1%})"})
    res = pd.DataFrame(rows); res.to_csv(f"{OUT}/hard151_check.csv", index=False, encoding="utf-8-sig")
    print(f"항상 맞힌 사진: {good.sum()}장"); print(res.to_string(index=False))